In [ ]:
import copy
import pprint
from collections import Counter
import torch
import torch.nn as nn
import wandb
import importlib
import src.cnn.cnn_utils as cnn_utils
import src.cnn.cnn_registry as cnn_registry
import src.cnn.cnn_models as cnn_models
import src.cnn.configs as configs
import src.cnn.cnn_paths as cnn_paths
import src.cnn.cnn_gradcam as cnn_gradcam


importlib.reload(cnn_utils)
#importlib.reload(cnn_gradcam)

In [ ]:
# see configs.py
cfg = configs.ExperimentConfig()

# =========================================================
# DATASET CONFIG
# =========================================================
cfg.dataset.dataset_dir = cnn_paths.DATASET_DIR  # see cnn_paths.py
cfg.dataset.train_subdir = "train"
cfg.dataset.val_subdir = "validate"
cfg.dataset.test_subdir = None  # or "test" if you have it
cfg.dataset.image_size = 224

# leave transforms as None -> default Resize + ToTensor pipeline
cfg.dataset.train_transform = None
cfg.dataset.eval_transform = None

# optional normalization
cfg.dataset.normalize_mean = None
cfg.dataset.normalize_std = None


# =========================================================
# MODEL CONFIG
# IMPORTANT: name is saved in MODEL_REGISTRY -> cnn_registry.py
# use @register_model("depth_cnn") decorator to add new models to the registry and make them available by name in the config
# =========================================================
cfg.model.name = "shallow_model"
cfg.model.kwargs = {
    "in_channels": 3,
    "num_classes": 10,
    "units": 128,
    "drop": 0.5,
}

# =========================================================
# TRAIN CONFIG
# =========================================================
cfg.train.epochs = 10
cfg.train.device = str(cnn_utils.get_device("auto"))
cfg = cnn_utils.configure_runtime_defaults(cfg)
cfg.train.non_blocking = True
cfg.train.use_amp = False

# RECOMMENDED FOR CUDA
# cfg.train.use_amp = True

cfg.train.grad_clip_norm = None
cfg.train.best_metric = "val/accuracy"
cfg.train.best_mode = "max"
cfg.train.seed = 19

# =========================================================
# DATALOADER CONFIG
# =========================================================
cfg.loader.batch_size = 4
cfg.loader.num_workers = 0
cfg.loader.pin_memory = (torch.device(cfg.train.device).type == "cuda")

# RECOMMENDED FOR CUDA
# cfg.loader.pin_memory = True

cfg.loader.train_shuffle = True
cfg.loader.eval_shuffle = False
cfg.loader.drop_last_train = False
cfg.loader.drop_last_eval = False


# =========================================================
# LOSS CONFIG
# =========================================================
cfg.loss.cls = nn.CrossEntropyLoss
cfg.loss.kwargs = {}


# =========================================================
# OPTIMIZER CONFIG
# =========================================================
cfg.optimizer.cls = torch.optim.Adam
cfg.optimizer.kwargs = {
    "lr": 1e-3,
    "weight_decay": 1e-4,
}


# =========================================================
# SCHEDULER CONFIG
# Example: Reduce LR when validation loss plateaus
# =========================================================
#cfg.scheduler.cls = torch.optim.lr_scheduler.ReduceLROnPlateau
cfg.scheduler.cls = None
cfg.scheduler.kwargs = {
    "mode": "min",
    "factor": 0.5,
    "patience": 2,
}
cfg.scheduler.step_metric = "val/loss"

# =========================================================
# W&B CONFIG
# =========================================================
cfg.wandb.enabled = True
cfg.wandb.project = "MPW-CNN"
cfg.wandb.entity = "MSE_DeLearn_SPR26"
cfg.wandb.mode = "online"  # "online", "offline", or "disabled" for no logging
cfg.wandb.log_confusion_matrix = True

# NOTE: use meaningful names for runs, so the difference is clear
cfg.wandb.run_name = "shallow_model"

# NOTE: use meaningful grouping, for example, by model or by task.
# E.g. task "Data augmentation" => experiments on light-mid-heavy augmentation
cfg.wandb.group = "shallow_model"
cfg.wandb.job_type = "train"

# NOTE: use meaningful tags to filter runs in UI
cfg.wandb.tags = ["cnn"]
cfg.wandb.notes = ""

cfg.wandb.log_epoch_metrics = True
cfg.wandb.log_every_n_epochs = 1

cfg.wandb.metric_allowlist = {
    "train/loss",
    "train/accuracy",
    "val/loss",
    "val/accuracy",
    "gap/accuracy",
    "gap/loss",
    "lr",
}

cfg.wandb.summary_allowlist = {
    "best_epoch",
    "best_metric_name",
    "best_metric_value",
    "train_final/loss",
    "train_final/accuracy",
    "val_final/loss",
    "val_final/accuracy",
}

cfg.wandb.watch_model = False
cfg.wandb.watch_log = "all"
cfg.wandb.watch_log_freq = 100

In [ ]:
datasets_dict = cnn_utils.load_datasets(cfg.dataset)

train_dataset = datasets_dict["train"]
val_dataset = datasets_dict["val"]
test_dataset = datasets_dict.get("test")

In [ ]:
train_loader = cnn_utils.make_train_loader(train_dataset, cfg)
val_loader = cnn_utils.make_eval_loader(val_dataset, cfg)

test_loader = None
if test_dataset is not None:
    test_loader = cnn_utils.make_eval_loader(test_dataset, cfg)

In [ ]:
# Model builder from config

model = cnn_registry.build_model(cfg.model)

print(model)
print(f"Trainable parameters: {cnn_utils.get_num_parameters(model):,}")

In [ ]:
model, history, result = cnn_utils.train_and_evaluate_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    cfg=cfg,
    run_name=cfg.wandb.run_name,
)

In [ ]:
pprint.pprint(result)

In [ ]:
import torch

device = torch.device(cfg.train.device)

correct, total, acc = cnn_utils.manual_acc_debug(model, val_loader, device)
print("manual acc = ", correct, total, acc)

In [ ]:
pprint.pprint(cfg)